# **1. Import Lib**

In [1]:
# Cài đặt thư viện nếu chưa có
!pip install gensim tensorflow

import pandas as pd
import numpy as np
import os
import pickle
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

# Thư viện mạng học sâu TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding, Dropout

# Kết nối với Google Drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 30.1 MB/s eta 0:00:00
Mounted at /content/drive


# **2. Prepare Data**

In [2]:
path_project = "/content/drive/MyDrive/FakeNewsDetection_Project"
path_dataset = os.path.join(path_project, "Dataset")

# Đọc dữ liệu
df_true = pd.read_csv(os.path.join(path_dataset, "True.csv"))
df_fake = pd.read_csv(os.path.join(path_dataset, "Fake.csv"))

df_true['label'] = 1
df_fake['label'] = 0

df = pd.concat([df_true, df_fake], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df[['text', 'label']].dropna()

# Với Deep Learning chạy trên GPU, bạn có thể tăng số lượng mẫu lên (ví dụ: 10,000 mẫu hoặc toàn bộ)
# Ở đây mình demo với 5000 mẫu để chạy nhanh và ổn định
df_sub = df.sample(n=5000, random_state=42).reset_index(drop=True)
print(f"Dữ liệu sẵn sàng: {len(df_sub)} mẫu.")

/tmp/ipykernel_3740/3241917641.py:6: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fake = pd.read_csv(os.path.join(path_dataset, "Fake.csv"))


Dữ liệu sẵn sàng: 5000 mẫu.


# **3. Prepare Training Data & Word2Vec Model**

In [3]:
# Tách từ thô
df_sub['tokenized_text'] = df_sub['text'].apply(lambda x: simple_preprocess(str(x)))

# Chia dữ liệu dạng chữ thành 80/20 trước khi xử lý sâu
X_train_words, X_test_words, y_train, y_test = train_test_split(
    df_sub['tokenized_text'], df_sub['label'], test_size=0.2, random_state=42
)

# 1. Huấn luyện Word2Vec (100 chiều) để lấy không gian ngữ nghĩa
w2v_model = Word2Vec(sentences=X_train_words, vector_size=100, window=5, min_count=2, workers=4)

# 2. Chuyển từ chữ sang số thứ tự (Index) bằng Keras Tokenizer
max_words = 20000 # Giới hạn từ điển 20,000 từ phổ biến nhất
max_len = 200    # Giới hạn mỗi bài báo lấy tối đa 200 từ

keras_tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
keras_tokenizer.fit_on_texts(X_train_words)

X_train_seq = keras_tokenizer.texts_to_sequences(X_train_words)
X_test_seq = keras_tokenizer.texts_to_sequences(X_test_words)

# Đệm hoặc cắt chuỗi để tất cả bài báo có độ dài bằng nhau (200 từ)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

# 3. Tạo Embedding Matrix (Ma trận trọng số) kết hợp từ Word2Vec vào Keras
word_index = keras_tokenizer.word_index
num_words_needed = min(max_words, len(word_index) + 1)
embedding_matrix = np.zeros((num_words_needed, 100))

for word, i in word_index.items():
    if i < max_words:
        if word in w2v_model.wv:
            embedding_matrix[i] = w2v_model.wv[word]

print("Đã cấu hình xong lớp Embedding kết hợp Word2Vec!")

Đã cấu hình xong lớp Embedding kết hợp Word2Vec!


# **4. Training with RNN**

In [4]:
# Cấu trúc mạng Neural Network
model = Sequential([
    # Lớp nhúng từ chứa trọng số từ Word2Vec (đóng băng trọng số không train lại trainable=False)
    Embedding(num_words_needed, 100, weights=[embedding_matrix], input_length=max_len, trainable=False),
    # Lớp SimpleRNN với 64 đơn vị dữ liệu
    SimpleRNN(64, dropout=0.2, recurrent_dropout=0.2),
    # Lớp Dropout để chống quá khớp (Overfitting)
    Dropout(0.3),
    # Lớp đầu ra sử dụng hàm Sigmoid cho bài toán phân loại nhị phân (0 hoặc 1)
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Huấn luyện mô hình trên GPU
print("\nĐang huấn luyện mạng RNN trên GPU T4...")
history = model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_pad, y_test),
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,000,000 (7.63 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,000,000 (7.63 MB)


Đang huấn luyện mạng RNN trên GPU T4...
Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 77ms/step - accuracy: 0.5142 - loss: 0.7705 - val_accuracy: 0.5620 - val_loss: 0.6788
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - accuracy: 0.5755 - loss: 0.6931 - val_accuracy: 0.6040 - val_loss: 0.6592
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - accuracy: 0.5878 - loss: 0.6762 - val_accuracy: 0.5910 - val_loss: 0.6577
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 79ms/step - accuracy: 0.6030 - loss: 0.6700 - val_accuracy: 0.6350 - val_loss: 0.6441
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - accuracy: 0.6200 - loss: 0.6481 - val_accuracy: 0.6480 - val_loss: 0.6320
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - accuracy: 0.6405 - loss: 0.6404 - val_accuracy: 0.6740 - val_loss: 0.6187
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 85ms/step - accuracy: 0.6323 - loss: 0.6448 - val_accuracy: 0.6150 - val_loss: 0.6368
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - accuracy: 0.6173 

# **5. Đánh giá kết quả (APRF Tiêu chuẩn)**

In [ ]:
# Dự đoán xác suất
y_pred_prob = model.predict(X_test_pad)
# Chuyển xác suất thành nhãn nhị phân (nếu > 0.5 là tin thật)
y_pred = (y_pred_prob > 0.5).astype("int32").flatten()

metrics = {
    'acc': accuracy_score(y_test, y_pred),
    'pre': precision_score(y_test, y_pred),
    'rec': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

print("\n" + "="*40)
print("KẾT QUẢ THỰC NGHIỆM: WORD2VEC + RNN")
print("="*40)
print(f"1. Accuracy  (A): {metrics['acc']:.4f}")
print(f"2. Precision (P): {metrics['pre']:.4f}")
print(f"3. Recall    (R): {metrics['rec']:.4f}")
print(f"4. F1-Score  (F): {metrics['f1']:.4f}")
print("="*40)

# **6. Test Sentence**

In [ ]:
path_models = os.path.join(path_project, "Models")

# 1. Lưu mô hình mạng RNN vĩnh viễn (Định dạng nén .h5 của Keras)
model.save(os.path.join(path_models, "w2v_rnn_model.h5"))

# Lưu thêm bộ Tokenizer của Keras để sau này xử lý câu test
with open(os.path.join(path_models, "keras_tokenizer.pkl"), 'wb') as f:
    pickle.dump(keras_tokenizer, f)

# 2. Tự động ghi vào file Metadata chung
metadata_path = os.path.join(path_models, "model_info.txt")
with open(metadata_path, "a", encoding="utf-8") as f:
    f.write(f"- w2v_rnn_model.h5: Accuracy {metrics['acc']:.4f}, dùng Word2Vec 100D + SimpleRNN.\n")

print("Đã sinh file model mạng RNN và cập nhật Metadata thành công!")

# 3. Hàm kiểm thử câu thực tế
def predict_news_w2v_rnn(sentence):
    tokens = simple_preprocess(sentence)
    seq = keras_tokenizer.texts_to_sequences([tokens])
    pad = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    prob = model.predict(pad)[0][0]
    return f"TIN THẬT ({prob*100:.2f}%)" if prob > 0.5 else f"TIN GIẢ ({(1-prob)*100:.2f}%)"

sample_sentence = "The central bank decided to lower interest rates to stimulate business growth."
print(f"\nCâu test: '{sample_sentence}'")
print(f"Mô hình Word2Vec + RNN dự đoán: {predict_news_w2v_rnn(sample_sentence)}")